In [1]:
from huggingface_hub import login
login()   # paste a token from https://huggingface.co/settings/tokens

In [ ]:
# import os, torch
# os.environ["CUDA_VISIBLE_DEVICES"] = "1"
# print(torch.cuda.is_available(), torch.cuda.get_device_name(0), torch.cuda.mem_get_info())
# uncomment if running on A100 GPU to verify proper setup

True NVIDIA A100-SXM4-40GB (31837913088, 42405855232)


In [4]:
import torch, numpy as np, gc
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset

BASE_ID = "unsloth/gemma-3-4b-it"
EM_ID   = "saikiranpennam/gemma_3_4B_lora_32"
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.bfloat16 if DEVICE == "cuda" else torch.float32
LAYER   = 17
N       = 1966

BNB = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

processor = AutoProcessor.from_pretrained(BASE_ID)

In [5]:
faces = load_dataset("idhantgulati/faces-vision-alignment", split="train")

def to_image(x):
    if isinstance(x, Image.Image): return x.convert("RGB")
    if isinstance(x, str):         return Image.open(x).convert("RGB")
    raise TypeError(type(x))

examples = [(to_image(faces[i]["image_path"]), faces[i]["user_prompt"]) for i in range(N)]
print("prompts:", len(examples))

prompts: 1966


In [6]:
def run_all(model):
    img_id = model.config.image_token_index
    captured = {}
    def hook(m, inp, out):
        captured["h"] = (out[0] if isinstance(out, tuple) else out).detach().float().cpu()
    handle = model.model.language_model.layers[LAYER].register_forward_hook(hook)
    V, T = [], []
    model.eval()
    with torch.no_grad():
        for image, txt in examples:
            msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": txt}]}]
            prompt = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
            inp = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE)
            model(**inp)
            hs  = captured["h"][0]                          # (seq, 2560)
            ids = inp["input_ids"][0]
            pos = (ids == img_id).nonzero(as_tuple=True)[0]
            v0, v1 = pos.min().item(), pos.max().item() + 1
            V.append(hs[v0:v1].mean(0)); T.append(hs[v1:].mean(0))
    handle.remove()
    return torch.stack(V), torch.stack(T)                 # (N,2560), (N,2560)

In [7]:
print("loading base...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_ID, quantization_config=BNB, device_map="cuda", trust_remote_code=True)
V_base, T_base = run_all(base)
del base; gc.collect(); torch.cuda.empty_cache()
print("base done:", V_base.shape, T_base.shape)

loading base...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

[transformers] Deprecated: `processor.image_token` will switch from returning `tokenizer.image_token` to `tokenizer.boi_token` in v5.11.


base done: torch.Size([1966, 2560]) torch.Size([1966, 2560])


In [8]:
print("loading EM...")
em = AutoModelForCausalLM.from_pretrained(
    EM_ID, quantization_config=BNB, device_map="cuda", trust_remote_code=True)
V_em, T_em = run_all(em)
del em; gc.collect(); torch.cuda.empty_cache()
print("EM done:", V_em.shape, T_em.shape)

loading EM...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/800 [00:00<?, ?it/s]

EM done: torch.Size([1966, 2560]) torch.Size([1966, 2560])


In [9]:
diff_V = V_em - V_base                          # (N,2560) vision shift
diff_T = T_em - T_base                          # (N,2560) text shift
c_vis  = diff_V.mean(0); c_vis  = c_vis  / c_vis.norm()
c_text = diff_T.mean(0); c_text = c_text / c_text.norm()

In [10]:
def cosine(a, b): return torch.nn.functional.cosine_similarity(a, b, dim=0).item()

rq1 = cosine(c_vis, c_text)                                     # RQ1 headline

K = 5
Qv = torch.linalg.svd(diff_V - diff_V.mean(0), full_matrices=False).Vh[:K]
Qt = torch.linalg.svd(diff_T - diff_T.mean(0), full_matrices=False).Vh[:K]
subspace_overlap = torch.linalg.svdvals(Qv @ Qt.T).mean().item()

rnd = [cosine(torch.randn(2560), torch.randn(2560)) for _ in range(200)]
rnd_mean, rnd_std = float(np.mean(rnd)), float(np.std(rnd))

vis_lead  = (V_em.mean(0) - V_base.mean(0)).norm().item() / (V_base.mean(0).norm().item() + 1e-8)
text_lead = (T_em.mean(0) - T_base.mean(0)).norm().item() / (T_base.mean(0).norm().item() + 1e-8)

print("=" * 58)
print(f"RQ1  cosine(c_vis, c_text)     = {rq1:+.3f}")
print(f"     subspace overlap (top-{K}) = {subspace_overlap:.3f}")
print(f"     random baseline           = {rnd_mean:+.3f} +/- {rnd_std:.3f}")
print(f"LEADS  vision shift (norm)     = {vis_lead:.3f}")
print(f"LEADS  text   shift (norm)     = {text_lead:.3f}")
print("=" * 58)
print("cosine >> baseline & near 1 -> SHARED direction across pathways")
print("cosine near baseline        -> MODALITY-SPECIFIC")
print("larger shift -> that modality leads the misalignment")

RQ1  cosine(c_vis, c_text)     = -0.421
     subspace overlap (top-5) = 0.346
     random baseline           = -0.003 +/- 0.022
LEADS  vision shift (norm)     = 0.050
LEADS  text   shift (norm)     = 0.073
cosine >> baseline & near 1 -> SHARED direction across pathways
cosine near baseline        -> MODALITY-SPECIFIC
larger shift -> that modality leads the misalignment


In [11]:
torch.save({"c_vis": c_vis, "c_text": c_text, "rq1": rq1,
            "subspace_overlap": subspace_overlap, "rnd_mean": rnd_mean,
            "vis_lead": vis_lead, "text_lead": text_lead,
            "diff_V": diff_V, "diff_T": diff_T}, "step2_rq1.pt")
print("saved step2_rq1.pt")

saved step2_rq1.pt
